### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as config
import deep_learning_land_use_classification.evaluation as evaluation
from deep_learning_land_use_classification.dataset import get_multi_label_data
import deep_learning_land_use_classification.wanddb_helpers as wh
from deep_learning_land_use_classification.early_stopping import EarlyStopper

# External imports
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb

In [2]:
train_df, test_df, val_df,class_names, num_classes = get_multi_label_data()

In [3]:
# Start a new wandb run to track this script.
run = wh.init_run(
    task="multi",
    architecture="resnet50",
    num_classes=num_classes,
    loss="BCEWithLogitsLoss",
    epochs=50,
    batch_size=32,
    learning_rate=1e-4,
    optimizer="Adam",
    pretrained=True,
    pretraining_dataset="ImageNetV2",
    pretraining_source="torchvision",
    weights="IMAGENET1K_V2",
    model_name=None,
    augmentation=False,
    early_stopping=True,
    patience=2,
    min_delta=0.001
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tomer\_netrc.
wandb: Currently logged in as: tomer-peled (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Resize, transform and normalize dataset

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # standard ImageNet mean
        std =[0.229, 0.224, 0.225] # standard ImageNet std
    )
])

### Get training and test dataset, as well as dataset loaders

In [5]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return image, label

In [6]:
train_dataset = MultiLabelDataset(train_df, class_names, transform)
val_dataset  = MultiLabelDataset(val_df, class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=run.config.batch_size, shuffle=False)

### Initiate model

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Using device: cuda


We use BCEWithLogitsLoss because our goal is multiclass classification. We also penalize larger classes using pos_weight due to the class imbalance present in the dataset. 

In [8]:
labels = train_df[class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

In [9]:
wh.log_model_summary(run, model)

### Train model

In [10]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [11]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
epochs = run.config.epochs
early_stopper = EarlyStopper(patience=run.config.patience, min_delta=run.config.min_delta)

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss  = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"val Loss:  {val_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_multilabel(
        model,
        val_loader,
        device,
        threshold=0.5
    )

    # Log metrics to Weights & Biases
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))

    wh.log_epoch(run, epoch, train_loss, val_loss,
        precision, recall, f1, p_macro, r_macro, f1_macro, class_names)
    
    # --- Early stopping check ---
    if early_stopper.step(val_loss, model):
        print(f"Early stopping triggered at epoch {epoch+1}. Best val loss: {early_stopper.best_loss:.4f}")
        break
    
# Restore the weights from the best epoch
early_stopper.restore_best_weights(model)
print("Restored best model weights.")
    
run.finish()
    

Epoch 1/50
Train Loss: 0.8638
val Loss:  0.5035
Epoch 2/50
Train Loss: 0.3901
val Loss:  0.2888
Epoch 3/50
Train Loss: 0.2242
val Loss:  0.2305
Epoch 4/50
Train Loss: 0.1562
val Loss:  0.2056
Epoch 5/50
Train Loss: 0.1220
val Loss:  0.1908
Epoch 6/50
Train Loss: 0.0988
val Loss:  0.1951
Epoch 7/50
Train Loss: 0.0812
val Loss:  0.1881
Epoch 8/50
Train Loss: 0.0629
val Loss:  0.1991
Epoch 9/50
Train Loss: 0.0525
val Loss:  0.2069
Early stopping triggered at epoch 9. Best val loss: 0.1881
Restored best model weights.


class/airplane/f1,▁████████
class/airplane/precision,▁████████
class/airplane/recall,▁▁▁▁▁▁▁▁▁
class/bare_soil/f1,▁▆▅▇████▇
class/bare_soil/precision,▁▅▅▇████▇
class/bare_soil/recall,▇▅█▃▃▅▃▁▃
class/buildings/f1,▁▄▆▆█▆▆█▇
class/buildings/precision,▁▄▅▆▇▆▆█▇
class/buildings/recall,█▄▆▄▅▃▂▃▁
class/cars/f1,▁▄▅▆▆▇▇█▇
+47,...
